# Fine-Tuning Basic Pitch for Guitar Audio-to-MIDI

**Goal:** specialize Basic Pitch on guitar data to push note-level (audio-to-MIDI) accuracy, while keeping it drop-in for the existing Fretwork pipeline.

**Recipe source:** TART (arXiv:2510.02597) Stage 1 + Riley et al. (ICASSP 2024) domain adaptation. Their proven result: fine-tuning on **GuitarSet + EGDB combined** ("Acoustic-Electric") beat every single-dataset variant on every test set, lifting GuitarSet note F50 from **0.704 (no fine-tune) -> 0.838**.

---
### What this notebook is
A complete fine-tuning pipeline: data prep -> segmentation -> augmentation -> target generation -> training loop -> P50/R50/F50 evaluation. The **data-prep and evaluation logic is concrete and unit-tested**; the **model-loading and training-loop sections are scaffolds wired to TART's proven config** that you point at a *trainable* Basic Pitch implementation.

### Honest prerequisites (read before running)
1. **A trainable Basic Pitch.** The pip `basic-pitch` package is inference-only. To fine-tune you need either:
   - the **PyTorch port** of Basic Pitch (model + pretrained weights exposed as `nn.Module`), **recommended** for ease of fine-tuning, or
   - Spotify's original **TensorFlow training code** in the `basic-pitch` repo.
   Confirm which you're using and adapt the two cells flagged `# >>> IMPL-SPECIFIC`.
2. **Data:** GuitarSet (audio + JAMS) and EGDB (audio + MIDI), on Drive.
3. **A GPU** (Colab T4/A100 is fine — the model is small; the cost is the 100k steps).

> If you'd rather fine-tune **Kong's** piano model the way TART/Riley did (the SOTA base), the data-prep, augmentation, and eval cells here are reusable as-is; only the model/target cells change.


In [4]:
!git clone https://github.com/spotify/basic-pitch.git /content/basic-pitch-read 2>/dev/null
with open('/content/basic-pitch-read/basic_pitch/train.py') as f:
    print(f.read())

#!/usr/bin/env python
# encoding: utf-8
#
# Copyright 2024 Spotify AB
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import argparse
import os
import logging
from datetime import datetime, timezone
from typing import List

import numpy as np
import tensorflow as tf

from basic_pitch import models
from basic_pitch.callbacks import VisualizeCallback
from basic_pitch.constants import DATASET_SAMPLING_FREQUENCY
from basic_pitch.data import tf_example_deserialization

logging.basicConfig(level=loggin

In [5]:
import os
print(os.listdir('/content/basic-pitch-read/basic_pitch/data'))

['datasets', '__init__.py', 'pipeline.py', 'tf_example_serialization.py', 'README.md', 'tf_example_deserialization.py', 'download.py', 'commandline.py']


In [7]:
import os
print(os.listdir('/content/basic-pitch-read/basic_pitch/data/datasets'))

['guitarset.py', '__init__.py', 'slakh.py', 'ikala.py', 'medleydb_pitch.py', 'maestro.py']


In [8]:
with open('/content/basic-pitch-read/basic_pitch/data/tf_example_deserialization.py') as f:
    print(f.read())

#!/usr/bin/env python
# encoding: utf-8
#
# Copyright 2024 Spotify AB
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import uuid
from typing import Any, Callable, Dict, Iterator, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# import tensorflow_addons as tfa

from basic_pitch.constants import (
    ANNOTATIONS_FPS,
    ANNOT_N_FRAMES,
    AUDIO_N_CHANNELS,
    AUDIO_N_SAMPLES,
    AUDIO_SAMPLE_RATE,
    AUDIO_WINDOW_LENGTH,
    N_FREQ_BINS_NOTES,
    N_FREQ_BINS_CO

In [9]:
with open('/content/basic-pitch-read/basic_pitch/data/tf_example_serialization.py') as f:
    print(f.read())

#!/usr/bin/env python
# encoding: utf-8
#
# Copyright 2022 Spotify AB
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from typing import List, Tuple, Union

import sox
import numpy as np
import tensorflow as tf
from basic_pitch.constants import AUDIO_N_CHANNELS, AUDIO_SAMPLE_RATE


def int64_feature(value: Union[List[int], int]) -> tf.train.Feature:
    if not isinstance(value, list):
        value = [value]
    return tf.train.Feature(int64_list=tf.train.Int64List(value=value))


def float_featu

In [10]:
with open('/content/basic-pitch-read/basic_pitch/data/datasets/guitarset.py') as f:
    print(f.read())

#!/usr/bin/env python
# encoding: utf-8
#
# Copyright 2024 Spotify AB
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import argparse
import logging
import os
import random
import time

from typing import Any, List, Dict, Tuple, Optional

import apache_beam as beam
import mirdata

from basic_pitch.data import commandline, pipeline


class GuitarSetInvalidTracks(beam.DoFn):
    def process(self, element: Tuple[str, str], *args: Tuple[Any, Any], **kwargs: Dict[str, Any]) -> Any:
        track_id, s

In [11]:
with open('/content/basic-pitch-read/basic_pitch/constants.py') as f:
    print(f.read())

#!/usr/bin/env python
# encoding: utf-8
#
# Copyright 2024 Spotify AB
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import numpy as np

from enum import Enum


SEMITONES_PER_OCTAVE = 12  # for frequency bin calculations

FFT_HOP = 256

NOTES_BINS_PER_SEMITONE = 1
CONTOURS_BINS_PER_SEMITONE = 3
# base frequency of the CENTRAL bin of the first semitone (i.e., the
# second bin if annotations_bins_per_semitone is 3)
ANNOTATIONS_BASE_FREQUENCY = 27.5  # lowest key on a piano
ANNOTATIONS_N_SEMITONES 

## 1. Setup

In [1]:
# Network ON required. In Colab:
# !pip -q install librosa pretty_midi jams mir_eval h5py soundfile tqdm
# # >>> IMPL-SPECIFIC: install your trainable Basic Pitch (PyTorch port recommended), e.g.:
# # !pip -q install basic-pitch-torch        # community PyTorch port (verify current name/API)
# # or clone Spotify's basic-pitch repo for the TF training pipeline.

import os, glob, math, random, json
import numpy as np
import pandas as pd
from dataclasses import dataclass, field

SEED = 0
random.seed(SEED); np.random.seed(SEED)
print("setup cell ran")

setup cell ran


In [2]:
# ── Mount Drive ──────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
except ModuleNotFoundError:
    print('Not running in Colab. Skipping mount.')

# ── Locate GuitarSet (same candidates as your other notebooks) ───────────────
import os, glob, json
from pathlib import Path

DATA_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Capstone/FullGuitarSetData'),
    Path('/content/drive/MyDrive/FullGuitarSetData'),
    Path('/content/drive/MyDrive/Capstone/GuitarSet'),
    Path('/content/drive/MyDrive/GuitarSet'),
]

DATA_ROOT = None
for c in DATA_ROOT_CANDIDATES:
    if (c / 'JamsFiles').exists():
        DATA_ROOT = c; break

if DATA_ROOT is None:
    raise FileNotFoundError(f"GuitarSet not found in any of: {DATA_ROOT_CANDIDATES}")

JAMS_DIR  = DATA_ROOT / 'JamsFiles'
AUDIO_DIR = DATA_ROOT / 'AudioFiles'

print(f"Data root : {DATA_ROOT}")
print(f"JAMS files: {len(list(JAMS_DIR.glob('*.jams')))}")
print(f"Audio files: {len(list(AUDIO_DIR.glob('*.wav')))}")

# ── Update cfg to use these paths ────────────────────────────────────────────
cfg.guitarset_jams_dir  = str(JAMS_DIR)
cfg.guitarset_audio_dir = str(AUDIO_DIR)
cfg.out_dir = '/content/drive/MyDrive/Capstone/outputs/finetune_basic_pitch'
os.makedirs(cfg.out_dir, exist_ok=True)
print(f"Checkpoints -> {cfg.out_dir}")

Mounted at /content/drive
Google Drive mounted successfully.
Data root : /content/drive/MyDrive/Capstone/FullGuitarSetData
JAMS files: 360
Audio files: 360


NameError: name 'cfg' is not defined

## 2. Config — TART's proven recipe (don't change without a reason)

In [4]:
@dataclass
class Cfg:
    # ---- paths (edit) ----
    guitarset_audio_dir: str = "/content/drive/MyDrive/fretwork/guitarset/audio_mono-mic"
    guitarset_jams_dir:  str = "/content/drive/MyDrive/fretwork/guitarset/annotation"
    egdb_audio_dir:      str = "/content/drive/MyDrive/fretwork/egdb/audio"
    egdb_midi_dir:       str = "/content/drive/MyDrive/fretwork/egdb/midi"
    out_dir:             str = "/content/drive/MyDrive/fretwork/finetune_runs"

    # ---- audio / features (must match the Basic Pitch impl you load) ----
    sample_rate: int = 22050
    fft_hop:     int = 256                 # Basic Pitch default -> ~86.13 frames/sec
    n_note_bins: int = 88                  # MIDI 21..108 (A0..C8)
    contour_bins_per_semitone: int = 3     # Basic Pitch contour head = 3*88 = 264
    midi_min:    int = 21

    # ---- segmentation + augmentation (TART) ----
    segment_sec: float = 10.0
    hop_sec:     float = 1.0
    pitch_shift_semitones = (-2, -1, 0, 1, 2)   # +/-2; onset-jitter intentionally OMITTED (TART: harmful)

    # ---- optimization (TART) ----
    batch_size:  int = 4
    lr:          float = 1e-5
    lr_decay:    float = 0.9               # x0.9 every decay_every steps
    decay_every: int = 10_000
    total_steps: int = 100_000
    val_every:   int = 2_000
    freeze_backbone: bool = False         # try True as a regularized ablation

    # ---- eval ----
    onset_tol_sec: float = 0.05           # +/-50 ms -> P50/R50/F50

    @property
    def fps(self): return self.sample_rate / self.fft_hop

cfg = Cfg()
print(f"frame rate = {cfg.fps:.2f} fps | contour bins = {cfg.contour_bins_per_semitone*cfg.n_note_bins}")

frame rate = 86.13 fps | contour bins = 264


In [6]:
!pip -q install jams pretty_midi librosa mir_eval soundfile h5py tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 64.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.1 MB/s eta 0:00:00


## 3. Data loading — GuitarSet (JAMS) + EGDB (MIDI), merged

For audio-to-MIDI we only need **(onset, offset, midi)** per note, collapsed across strings. We reuse the existing GuitarSet JAMS parsing convention and add a `pretty_midi` reader for EGDB.

In [7]:
import json, pretty_midi

def load_guitarset_notes(jams_path):
    """Read GuitarSet JAMS as raw JSON; merge note_midi across all 6 strings."""
    with open(jams_path) as f:
        jam = json.load(f)
    notes = []
    for ann in jam.get("annotations", []):
        ns = ann.get("namespace", "")
        if ns not in ("note_midi", "pitch_midi"):
            continue
        for obs in ann["data"]:
            notes.append({"onset": float(obs["time"]),
                          "offset": float(obs["time"]) + float(obs["duration"]),
                          "midi": int(round(float(obs["value"])))})
    return sorted(notes, key=lambda n: (n["onset"], n["midi"]))

def load_midi_notes(midi_path):
    """EGDB / any MIDI -> merged note list."""
    pm = pretty_midi.PrettyMIDI(midi_path)
    notes = []
    for inst in pm.instruments:
        if inst.is_drum: continue
        for n in inst.notes:
            notes.append({"onset": float(n.start), "offset": float(n.end), "midi": int(n.pitch)})
    return sorted(notes, key=lambda n: (n["onset"], n["midi"]))

def build_index(cfg):
    """Pair audio files with their note lists. Returns list of records with a 'source' tag."""
    recs = []
    for jp in sorted(glob.glob(os.path.join(cfg.guitarset_jams_dir, "*.jams"))):
        stem = os.path.splitext(os.path.basename(jp))[0]
        # GuitarSet mic files are usually <stem>_mic.wav — adjust to your naming
        cands = glob.glob(os.path.join(cfg.guitarset_audio_dir, stem + "*mic*.wav")) \
                or glob.glob(os.path.join(cfg.guitarset_audio_dir, stem + "*.wav"))
        if cands:
            recs.append({"id": stem, "audio": cands[0], "jams": jp, "source": "guitarset"})
    for mp in sorted(glob.glob(os.path.join(cfg.egdb_midi_dir, "*.mid*"))):
        stem = os.path.splitext(os.path.basename(mp))[0]
        cands = glob.glob(os.path.join(cfg.egdb_audio_dir, stem + "*.wav"))
        if cands:
            recs.append({"id": stem, "audio": cands[0], "midi": mp, "source": "egdb"})
    return recs

def notes_for(rec):
    return load_guitarset_notes(rec["jams"]) if rec["source"]=="guitarset" else load_midi_notes(rec["midi"])

# records = build_index(cfg)
# print(pd.Series([r["source"] for r in records]).value_counts())

## 4. Splits — player-disjoint (GuitarSet) so a player never leaks across train/test

Mirrors the paper's leave-one-guitarist-out discipline. EGDB has no player metadata, so split it by recording. Define the variants from the TART table so you can reproduce Base / Acoustic / Acoustic-Electric.

In [8]:
def guitarset_player(rec_id):
    # GuitarSet ids look like "00_BN1-129-Eb_comp" -> player prefix "00".."05"
    return rec_id.split("_")[0]

def make_splits(records, held_out_players=("05",), val_frac=0.2, seed=0):
    rng = random.Random(seed)
    gs = [r for r in records if r["source"]=="guitarset"]
    eg = [r for r in records if r["source"]=="egdb"]
    test  = [r for r in gs if guitarset_player(r["id"]) in held_out_players]
    trainable_gs = [r for r in gs if guitarset_player(r["id"]) not in held_out_players]
    rng.shuffle(eg)
    n_eg_val = int(len(eg)*val_frac)
    rng.shuffle(trainable_gs)
    n_gs_val = int(len(trainable_gs)*val_frac)
    splits = {
        "guitarset_train": trainable_gs[n_gs_val:], "guitarset_val": trainable_gs[:n_gs_val],
        "egdb_train": eg[n_eg_val:], "egdb_val": eg[:n_eg_val],
        "test_guitarset": test,
    }
    return splits

# Experiment variants (TART Tables 2-4)
def variant_records(splits, variant):
    if variant=="acoustic":            # GuitarSet only
        return splits["guitarset_train"], splits["guitarset_val"]
    if variant=="electric":            # EGDB only
        return splits["egdb_train"], splits["egdb_val"]
    if variant=="acoustic_electric":   # combined (TART's winner)
        return splits["guitarset_train"]+splits["egdb_train"], splits["guitarset_val"]+splits["egdb_val"]
    raise ValueError(variant)

## 5. Segmentation + pitch-shift augmentation  ✅ *unit-tested logic*

10 s windows, 1 s hop. Pitch-shift transforms **audio and labels together** (so labels stay exact). Onset jitter is deliberately excluded (TART found it harmful).

In [9]:
def segment_bounds(duration, seg=10.0, hop=1.0):
    out=[]; t=0.0
    while t < duration:
        out.append((t, min(t+seg, duration)))
        if t+seg >= duration: break
        t += hop
    return out

def notes_in_segment(notes, t0, t1):
    """Clip notes to [t0,t1), re-zero times to segment start."""
    seg=[]
    for n in notes:
        if n["onset"] < t1 and n["offset"] > t0:
            seg.append({"onset": max(0.0, n["onset"]-t0),
                        "offset": min(t1, n["offset"])-t0,
                        "midi": n["midi"]})
    return seg

def shift_notes(notes, semis):
    return [{**n, "midi": n["midi"]+semis} for n in notes]

# ---- self-test ----
_gt=[{"onset":0.0,"offset":0.4,"midi":60},{"onset":0.5,"offset":0.9,"midi":64},{"onset":1.0,"offset":1.4,"midi":67}]
assert len(segment_bounds(12.0))==3
assert [n["midi"] for n in shift_notes(_gt,2)]==[62,66,69]
assert [n["midi"] for n in notes_in_segment(_gt,0.5,1.5)]==[64,67]
print("segmentation + augmentation logic OK")

def pitch_shift_audio(y, sr, semis):
    """librosa pitch shift; pair with shift_notes(notes, semis)."""
    import librosa
    return y if semis==0 else librosa.effects.pitch_shift(y, sr=sr, n_steps=semis)

segmentation + augmentation logic OK


## 6. Target generation — Basic Pitch's onset / note / contour posteriorgrams

Basic Pitch trains three 2-D targets per segment: **onset** (T×88), **note** (T×88), **contour** (T×264). Built from the segment's note events at the model's frame rate.

> `# >>> IMPL-SPECIFIC` The exact bin layout / Gaussian-smoothing of targets must match the implementation you load. Constants below are Basic Pitch defaults; verify against the port's data code.

In [10]:
def time_to_frame(t, cfg): return int(round(t * cfg.fps))

def notes_to_targets(notes, n_frames, cfg, onset_spread=1):
    """Return (onset, note, contour) float arrays. Minimal/standard construction."""
    n_note = cfg.n_note_bins
    n_cont = cfg.contour_bins_per_semitone * n_note
    onset  = np.zeros((n_frames, n_note), np.float32)
    note   = np.zeros((n_frames, n_note), np.float32)
    contour= np.zeros((n_frames, n_cont), np.float32)
    for nt in notes:
        b = nt["midi"] - cfg.midi_min
        if not (0 <= b < n_note): continue
        f0 = time_to_frame(nt["onset"], cfg)
        f1 = max(f0+1, time_to_frame(nt["offset"], cfg))
        f0c, f1c = max(0,f0), min(n_frames, f1)
        note[f0c:f1c, b] = 1.0
        for df in range(-onset_spread, onset_spread+1):     # small onset window
            fi=f0+df
            if 0<=fi<n_frames: onset[fi, b]=max(onset[fi,b], 1.0-0.3*abs(df))
        cb = b*cfg.contour_bins_per_semitone + cfg.contour_bins_per_semitone//2
        if 0<=cb<n_cont: contour[f0c:f1c, cb]=1.0
    return onset, note, contour

# sanity: a single A4 (midi 69) for ~0.5s
_o,_n,_c = notes_to_targets([{"onset":0.1,"offset":0.6,"midi":69}], n_frames=60, cfg=cfg)
assert _n[:, 69-cfg.midi_min].sum() > 0 and _o[:, 69-cfg.midi_min].sum() > 0
print("target generation OK | shapes:", _o.shape, _n.shape, _c.shape)

target generation OK | shapes: (60, 88) (60, 88) (60, 264)


## 7. Dataset / DataLoader

Materializes (audio segment -> input features, targets). Loading audio per segment is simplest; for speed, pre-cache features to HDF5 (TART used a unified HDF5).

> `# >>> IMPL-SPECIFIC` `audio_to_model_input` must produce exactly what the model's forward expects (Basic Pitch uses a Harmonic-CQT stack computed inside the package; reuse the port's featurizer rather than rolling your own).

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader

class GuitarFinetuneDataset(Dataset):
    def __init__(self, records, cfg, augment=True):
        self.cfg=cfg; self.augment=augment
        self.items=[]   # (record, t0, t1)
        for r in records:
            import soundfile as sf
            dur = sf.info(r["audio"]).duration
            for (t0,t1) in segment_bounds(dur, cfg.segment_sec, cfg.hop_sec):
                if t1-t0 >= 1.0: self.items.append((r,t0,t1))
        self._notes_cache={}

    def __len__(self): return len(self.items)

    def _notes(self, r):
        if r["id"] not in self._notes_cache: self._notes_cache[r["id"]]=notes_for(r)
        return self._notes_cache[r["id"]]

    def __getitem__(self, i):
        import librosa
        r,t0,t1 = self.items[i]; cfg=self.cfg
        y,_ = librosa.load(r["audio"], sr=cfg.sample_rate, offset=t0, duration=t1-t0)
        notes = notes_in_segment(self._notes(r), t0, t1)
        semis = random.choice(cfg.pitch_shift_semitones) if self.augment else 0
        if semis: y = pitch_shift_audio(y, cfg.sample_rate, semis); notes = shift_notes(notes, semis)
        n_frames = int(round((t1-t0)*cfg.fps))
        onset,note,contour = notes_to_targets(notes, n_frames, cfg)
        x = audio_to_model_input(y, cfg)                      # >>> IMPL-SPECIFIC
        return {"x":x,
                "onset":torch.from_numpy(onset), "note":torch.from_numpy(note),
                "contour":torch.from_numpy(contour)}

def audio_to_model_input(y, cfg):
    # >>> IMPL-SPECIFIC: replace with the port's HCQT featurizer (or return raw audio if the model takes audio).
    raise NotImplementedError("Wire to your Basic Pitch implementation's input featurizer.")

In [12]:
_n = load_guitarset_notes(glob.glob(os.path.join(cfg.guitarset_jams_dir, "*.jams"))[0])
print(len(_n), "notes |", min(x["midi"] for x in _n), "-", max(x["midi"] for x in _n))

368 notes | 40 - 70


## 8. Model — load pretrained Basic Pitch for fine-tuning

In [13]:
!git clone https://github.com/spotify/basic-pitch.git /content/basic-pitch
!pip -q install -e "/content/basic-pitch[tf]"

Cloning into '/content/basic-pitch'...
remote: Enumerating objects: 1657, done.
remote: Counting objects: 100% (831/831), done.
remote: Compressing objects: 100% (337/337), done.
remote: Total 1657 (delta 638), reused 494 (delta 494), pack-reused 826 (from 3)
Receiving objects: 100% (1657/1657), 282.68 MiB | 25.69 MiB/s, done.
Resolving deltas: 100% (947/947), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement tensorflow<2.15.1,>=2.4.1; platform_system != "Darwin" and python_version >= "3.11" (from basic-pitch) (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflo

In [15]:
!pip -q install -e /content/basic-pitch --no-deps
!pip -q install tensorflow librosa resampy pretty_midi

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for basic-pitch (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
basic-pitch 0.4.0 requires resampy<0.4.3,>=0.2.2, but you have resampy 0.4.3 which is incompatible.
basic-pitch 0.4.0 requires tensorflow<2.15.1,>=2.4.1; platform_system != "Darwin" and python_version >= "3.11", but you have tensorflow 2.20.0 which is incompatible.


In [17]:
import basic_pitch, os
print(basic_pitch.__file__)
print(os.listdir(os.path.dirname(basic_pitch.__file__)))

/content/basic-pitch/basic_pitch/__init__.py
['saved_models', 'commandline_printing.py', '__init__.py', 'layers', 'predict.py', 'inference.py', '__pycache__', 'nn.py', 'callbacks.py', 'constants.py', 'visualize.py', 'train.py', 'models.py', 'note_creation.py', 'data']


In [18]:
from basic_pitch import ICASSP_2022_MODEL_PATH
print("model path:", ICASSP_2022_MODEL_PATH)

import tensorflow as tf
model = tf.saved_model.load(str(ICASSP_2022_MODEL_PATH))
print("loaded OK")
print(list(model.signatures.keys()))

model path: /content/basic-pitch/basic_pitch/saved_models/icassp_2022/nmp
loaded OK
['serving_default']


In [19]:
# What does the serving signature expect and return?
infer = model.signatures['serving_default']
print("INPUTS:")
for k,v in infer.structured_input_signature[1].items():
    print(f"  {k}: {v}")
print("OUTPUTS:")
for k,v in infer.structured_outputs.items():
    print(f"  {k}: {v}")

INPUTS:
  input_2: TensorSpec(shape=(None, 43844, 1), dtype=tf.float32, name='input_2')
OUTPUTS:
  onset: TensorSpec(shape=(None, 172, 88), dtype=tf.float32, name='onset')
  contour: TensorSpec(shape=(None, 172, 264), dtype=tf.float32, name='contour')
  note: TensorSpec(shape=(None, 172, 88), dtype=tf.float32, name='note')


In [20]:
# What's in train.py and models.py?
!head -80 /content/basic-pitch/basic_pitch/train.py

#!/usr/bin/env python
# encoding: utf-8
#
# Copyright 2024 Spotify AB
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import argparse
import os
import logging
from datetime import datetime, timezone
from typing import List

import numpy as np
import tensorflow as tf

from basic_pitch import models
from basic_pitch.callbacks import VisualizeCallback
from basic_pitch.constants import DATASET_SAMPLING_FREQUENCY
from basic_pitch.data import tf_example_deserialization

logging.basicConfig(level=loggin

In [21]:
!head -60 /content/basic-pitch/basic_pitch/models.py

#!/usr/bin/env python
# encoding: utf-8
#
# Copyright 2022 Spotify AB
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from typing import Any, Callable, Dict
import numpy as np
import tensorflow as tf

from basic_pitch import nn
from basic_pitch.constants import (
    ANNOTATIONS_BASE_FREQUENCY,
    ANNOTATIONS_N_SEMITONES,
    AUDIO_N_SAMPLES,
    AUDIO_SAMPLE_RATE,
    CONTOURS_BINS_PER_SEMITONE,
    FFT_HOP,
    N_FREQ_BINS_CONTOURS,
)
from basic_pitch.layers import signal, nnaudio

tfkl = tf.k

In [22]:
import sys
sys.path.insert(0, '/content/basic-pitch')

from basic_pitch.constants import (
    AUDIO_N_SAMPLES,      # 43844 — samples per chunk at 22050 Hz
    AUDIO_SAMPLE_RATE,    # 22050
    FFT_HOP,              # 256
    ANNOTATIONS_N_SEMITONES,  # 88
    N_FREQ_BINS_CONTOURS,     # 264
    ANNOTATIONS_BASE_FREQUENCY,
)
from basic_pitch.models import transcription_loss
import numpy as np

# Derived constants
N_FRAMES = 172   # frames per 2s chunk (matches model output shape)
MIDI_MIN = 21    # A0

print(f"chunk size: {AUDIO_N_SAMPLES} samples = {AUDIO_N_SAMPLES/AUDIO_SAMPLE_RATE:.2f}s")
print(f"output frames per chunk: {N_FRAMES}")
print(f"note bins: {ANNOTATIONS_N_SEMITONES} | contour bins: {N_FREQ_BINS_CONTOURS}")

def chunk_bounds(duration, chunk_samples=AUDIO_N_SAMPLES, sr=AUDIO_SAMPLE_RATE, hop_sec=1.0):
    """Split audio into fixed 2s chunks with 1s hop (TART recipe adapted to BP's window size)."""
    chunk_sec = chunk_samples / sr
    out = []; t = 0.0
    while t < duration:
        out.append((t, min(t + chunk_sec, duration)))
        if t + chunk_sec >= duration: break
        t += hop_sec
    return out

def notes_to_targets(notes, n_frames=N_FRAMES, midi_min=MIDI_MIN,
                     n_note=ANNOTATIONS_N_SEMITONES, n_cont=N_FREQ_BINS_CONTOURS,
                     sr=AUDIO_SAMPLE_RATE, hop=FFT_HOP):
    """Build onset/note/contour target arrays from a note list (times in seconds, re-zeroed to chunk start)."""
    fps = sr / hop
    onset  = np.zeros((n_frames, n_note), np.float32)
    note   = np.zeros((n_frames, n_note), np.float32)
    contour= np.zeros((n_frames, n_cont), np.float32)
    cps = N_FREQ_BINS_CONTOURS // n_note   # contour bins per semitone = 3
    for nt in notes:
        b = nt['midi'] - midi_min
        if not (0 <= b < n_note): continue
        f0 = int(round(nt['onset'] * fps))
        f1 = max(f0 + 1, int(round(nt['offset'] * fps)))
        f0c, f1c = max(0, f0), min(n_frames, f1)
        note[f0c:f1c, b] = 1.0
        if 0 <= f0 < n_frames: onset[f0, b] = 1.0
        cb = b * cps + cps // 2
        if 0 <= cb < n_cont: contour[f0c:f1c, cb] = 1.0
    return onset, note, contour

# quick self-test
_o, _n, _c = notes_to_targets([{'onset':0.1,'offset':0.6,'midi':69}])
assert _n[:, 69-MIDI_MIN].sum() > 0
print("target generation OK | shapes:", _o.shape, _n.shape, _c.shape)

chunk size: 43844 samples = 1.99s
output frames per chunk: 172
note bins: 88 | contour bins: 264
target generation OK | shapes: (172, 88) (172, 88) (172, 264)


In [23]:
import librosa, tensorflow as tf, random

def load_chunks(rec, cfg, augment=True):
    """Generator: yields (audio_chunk, onset, note, contour) for one recording."""
    notes_all = notes_for(rec)
    y, _ = librosa.load(rec['audio'], sr=AUDIO_SAMPLE_RATE, mono=True)
    duration = len(y) / AUDIO_SAMPLE_RATE
    semitones = list(cfg.pitch_shift_semitones) if augment else [0]
    for t0, t1 in chunk_bounds(duration):
        s0 = int(t0 * AUDIO_SAMPLE_RATE)
        s1 = s0 + AUDIO_N_SAMPLES
        chunk = y[s0:s1]
        if len(chunk) < AUDIO_N_SAMPLES:   # pad last chunk
            chunk = np.pad(chunk, (0, AUDIO_N_SAMPLES - len(chunk)))
        seg_notes = notes_in_segment(notes_all, t0, t0 + AUDIO_N_SAMPLES / AUDIO_SAMPLE_RATE)
        for semi in (random.sample(semitones, 1) if augment else [0]):
            c = librosa.effects.pitch_shift(chunk, sr=AUDIO_SAMPLE_RATE, n_steps=semi) if semi else chunk
            ns = shift_notes(seg_notes, semi) if semi else seg_notes
            onset, note, contour = notes_to_targets(ns)
            yield (c.reshape(-1, 1).astype(np.float32),
                   onset.astype(np.float32),
                   note.astype(np.float32),
                   contour.astype(np.float32))

def build_tf_dataset(records, cfg, augment=True, shuffle=True):
    def gen():
        recs = list(records)
        if shuffle: random.shuffle(recs)
        for r in recs:
            yield from load_chunks(r, cfg, augment=augment)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec((AUDIO_N_SAMPLES, 1), tf.float32),  # audio
            tf.TensorSpec((N_FRAMES, 88),  tf.float32),        # onset
            tf.TensorSpec((N_FRAMES, 88),  tf.float32),        # note
            tf.TensorSpec((N_FRAMES, 264), tf.float32),        # contour
        )
    )
    if shuffle: ds = ds.shuffle(256)
    return ds.batch(cfg.batch_size).prefetch(tf.data.AUTOTUNE)

print("tf.data pipeline ready")

tf.data pipeline ready


In [25]:
print([x for x in dir(bp_models) if not x.startswith('_')])

['ANNOTATIONS_BASE_FREQUENCY', 'ANNOTATIONS_N_SEMITONES', 'AUDIO_N_SAMPLES', 'AUDIO_SAMPLE_RATE', 'Any', 'CONTOURS_BINS_PER_SEMITONE', 'CONTOUR_FILTERS_2', 'CONTOUR_KERNEL_SIZE_1', 'CONTOUR_KERNEL_SIZE_2', 'CONTOUR_KERNEL_SIZE_3', 'Callable', 'DEFAULT_LABEL_SMOOTHING', 'DEFAULT_POSITIVE_WEIGHT', 'Dict', 'FFT_HOP', 'MAX_N_SEMITONES', 'NOTES_KERNEL_SIZE_1', 'NOTES_KERNEL_SIZE_2', 'NOTES_STRIDES_1', 'N_FREQ_BINS_CONTOURS', 'ONSET_KERNEL_SIZE_1', 'ONSET_KERNEL_SIZE_2', 'ONSET_STRIDES_1', 'get_cqt', 'loss', 'model', 'nn', 'nnaudio', 'np', 'onset_loss', 'signal', 'tf', 'tfkl', 'transcription_loss', 'weighted_transcription_loss']


In [26]:
from basic_pitch import models as bp_models

# Build the Keras model (same architecture as the SavedModel)
keras_model = bp_models.model()
keras_model.summary(line_length=100)

# Load pretrained weights from the SavedModel into the Keras model
saved = tf.saved_model.load(str(ICASSP_2022_MODEL_PATH))
# Extract weights via the serving signature and assign to Keras model
keras_model.set_weights([v.numpy() for v in saved.variables])
print("\nPretrained weights loaded into Keras model")

# Optional: freeze backbone, train heads only (regularized ablation)
if cfg.freeze_backbone:
    for layer in keras_model.layers:
        if not any(h in layer.name for h in ['onset', 'note', 'contour']):
            layer.trainable = False
    print("Backbone frozen")

trainable = sum(np.prod(v.shape) for v in keras_model.trainable_variables)
print(f"Trainable params: {trainable:,}")

AttributeError: 'tuple' object has no attribute 'rank'

In [27]:
# Use the SavedModel as-is and wrap it for fine-tuning
from basic_pitch import ICASSP_2022_MODEL_PATH

# Load as a TF module (already done earlier, but reloading cleanly here)
saved_model = tf.saved_model.load(str(ICASSP_2022_MODEL_PATH))

# Wrap in a Keras model so we get .fit(), gradient tape, etc.
class BasicPitchFinetune(tf.keras.Model):
    def __init__(self, saved_model):
        super().__init__()
        self.bp = saved_model

    def call(self, audio, training=False):
        # saved_model expects (batch, 43844, 1)
        return self.bp.signatures['serving_default'](input_2=audio)

keras_model = BasicPitchFinetune(saved_model)

# Quick forward pass to confirm shapes
test_audio = tf.zeros((1, AUDIO_N_SAMPLES, 1), dtype=tf.float32)
out = keras_model(test_audio)
print("onset:", out['onset'].shape)
print("note: ", out['note'].shape)
print("contour:", out['contour'].shape)
print("model wrapper OK")

onset: (1, 172, 88)
note:  (1, 172, 88)
contour: (1, 172, 264)
model wrapper OK


In [28]:
print(f"Total variables: {len(keras_model.bp.variables)}")
print(f"First few: {[v.name for v in keras_model.bp.variables[:3]]}")

Total variables: 24
First few: ['batch_normalization/gamma:0', 'batch_normalization/beta:0', 'batch_normalization/moving_mean:0']


In [29]:
# Find the callable TF functions inside the SavedModel
print(dir(saved_model))

['__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_add_trackable_child', '_add_variable_with_custom_getter', '_checkpoint_adapter', '_checkpoint_dependencies', '_copy_trackable_to_cpu', '_default_save_signature', '_deferred_dependencies', '_delete_tracking', '_deserialization_dependencies', '_deserialize_from_proto', '_export_to_saved_model_graph', '_gather_saveables_for_checkpoint', '_handle_deferred_dependencies', '_lookup_dependency', '_maybe_initialize_trackable', '_name_based_attribute_restore', '_name_based_restores', '_no_dependency', '_object_identifier', '_preload_simple_restoration', '_restore_from_tensors', '_self_name_based_restores', '_self_saveable_objec

In [30]:
# Check if there's a direct callable
print(type(saved_model))
try:
    out = saved_model(test_audio, training=False)
    print("direct call works:", {k:v.shape for k,v in out.items()})
except Exception as e:
    print("direct call failed:", e)

<class 'tensorflow.python.saved_model.load.Loader._recreate_base_user_object.<locals>._UserObject'>
direct call works: {'onset': TensorShape([1, 172, 88]), 'contour': TensorShape([1, 172, 264]), 'note': TensorShape([1, 172, 88])}


In [31]:
from basic_pitch import ICASSP_2022_MODEL_PATH

saved_model = tf.saved_model.load(str(ICASSP_2022_MODEL_PATH))

# Confirm direct call + gradient access
test_audio = tf.zeros((1, AUDIO_N_SAMPLES, 1), dtype=tf.float32)
out = saved_model(test_audio, training=False)
print("Forward pass OK:", {k: v.shape for k, v in out.items()})
print(f"Trainable variables: {len(saved_model.trainable_variables)}")

# Confirm gradients flow through direct call
with tf.GradientTape() as tape:
    out2 = saved_model(test_audio, training=True)
    test_loss = tf.reduce_mean(out2['onset'])
grads = tape.gradient(test_loss, saved_model.trainable_variables)
n_grads = sum(1 for g in grads if g is not None)
print(f"Gradient check: {n_grads}/{len(saved_model.trainable_variables)} variables have gradients")

if cfg.freeze_backbone:
    # Freeze all but the last few layers (onset/note/contour heads)
    for v in saved_model.trainable_variables[:-6]:
        v._trainable = False
    print("Backbone frozen")

Forward pass OK: {'onset': TensorShape([1, 172, 88]), 'contour': TensorShape([1, 172, 264]), 'note': TensorShape([1, 172, 88])}
Trainable variables: 18
Gradient check: 18/18 variables have gradients


In [49]:
import shutil, os, glob
from tqdm import tqdm

LOCAL_AUDIO = '/content/guitarset_audio'
LOCAL_JAMS  = '/content/guitarset_jams'
os.makedirs(LOCAL_AUDIO, exist_ok=True)
os.makedirs(LOCAL_JAMS,  exist_ok=True)

audio_files = glob.glob(os.path.join(cfg.guitarset_audio_dir, '*.wav'))
jams_files  = glob.glob(os.path.join(cfg.guitarset_jams_dir,  '*.jams'))

print(f"Copying {len(audio_files)} audio files...")
for f in tqdm(audio_files):
    shutil.copy2(f, LOCAL_AUDIO)

print(f"Copying {len(jams_files)} jams files...")
for f in tqdm(jams_files):
    shutil.copy2(f, LOCAL_JAMS)

print("Done. Updating cfg paths...")
cfg.guitarset_audio_dir = LOCAL_AUDIO
cfg.guitarset_jams_dir  = LOCAL_JAMS
print("cfg updated to local SSD paths")

Copying 360 audio files...


100%|██████████| 360/360 [00:52<00:00,  6.84it/s]


Copying 360 jams files...


100%|██████████| 360/360 [00:10<00:00, 34.15it/s]

Done. Updating cfg paths...
cfg updated to local SSD paths


In [50]:
records = build_index(cfg)
splits  = make_splits(records, held_out_players=("05",))
train_recs, val_recs = variant_records(splits, "acoustic")
print(f"Train: {len(train_recs)} | Val: {len(val_recs)} | Test: {len(splits['test_guitarset'])}")

Train: 240 | Val: 60 | Test: 60


In [53]:
cfg.guitarset_audio_dir = '/content/guitarset_audio'
cfg.guitarset_jams_dir  = '/content/guitarset_jams'

records = build_index(cfg)
splits  = make_splits(records, held_out_players=("05",))
train_recs, val_recs = variant_records(splits, "acoustic")
print(f"Train: {len(train_recs)} | Val: {len(val_recs)} | Test: {len(splits['test_guitarset'])}")

Train: 240 | Val: 60 | Test: 60


In [56]:
import time
print("Testing data pipeline...")
t0 = time.time()

train_ds_test = build_tf_dataset(train_recs[:5], cfg, augment=False, shuffle=False)
batch = next(iter(train_ds_test))
print(f"First batch loaded in {time.time()-t0:.1f}s")
print("audio:", batch[0].shape)
print("onset:", batch[1].shape)

Testing data pipeline...
First batch loaded in 0.7s
audio: (4, 43844, 1)
onset: (4, 172, 88)


In [57]:
import time
print("Testing single train step...")
audio, t_on, t_note, t_cont = next(iter(build_tf_dataset(train_recs[:2], cfg, augment=False, shuffle=False)))
t0 = time.time()
print("Running train_step (may take 2-5 min for first @tf.function trace)...")
loss = train_step(audio, t_on, t_note, t_cont)
print(f"First train step done in {time.time()-t0:.1f}s | loss: {float(loss):.4f}")

t0 = time.time()
loss = train_step(audio, t_on, t_note, t_cont)
print(f"Second train step done in {time.time()-t0:.1f}s | loss: {float(loss):.4f}")

Testing single train step...
Running train_step (may take 2-5 min for first @tf.function trace)...
First train step done in 3.0s | loss: 0.9821
Second train step done in 1.7s | loss: 0.9821


In [ ]:
# >>> IMPL-SPECIFIC ----------------------------------------------------------
# Load the PyTorch-port model WITH pretrained weights, expose 3 heads
# (onset, note, contour). Pseudocode:
#
# from basic_pitch_torch.model import BasicPitchTorch
# model = BasicPitchTorch()
# model.load_state_dict(torch.load("basic_pitch_pretrained.pt"))
#
# def forward_heads(model, x):
#     out = model(x)                 # dict or tuple
#     return out["onset"], out["note"], out["contour"]
#
# Optional regularized ablation: freeze the conv backbone, train heads only.
# if cfg.freeze_backbone:
#     for p in model.backbone.parameters(): p.requires_grad = False
# ---------------------------------------------------------------------------
print("Wire model loading to your trainable Basic Pitch, then expose forward_heads(model, x).")

## 9. Training loop — TART optimization recipe

Adam @ 1e-5, ×0.9 every 10k steps, 100k steps, BCE on all three heads. Periodic val + checkpoint-on-best.

In [63]:
from basic_pitch.models import DEFAULT_LABEL_SMOOTHING, transcription_loss
print("DEFAULT_LABEL_SMOOTHING:", DEFAULT_LABEL_SMOOTHING)

from basic_pitch.models import DEFAULT_LABEL_SMOOTHING, transcription_loss

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=cfg.lr,
    decay_steps=cfg.decay_every,
    decay_rate=cfg.lr_decay,
    staircase=True
)
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
print(f"Optimizer ready | initial lr: {cfg.lr} | decay: {cfg.lr_decay} every {cfg.decay_every} steps")


@tf.function(input_signature=[
    tf.TensorSpec(shape=(None, AUDIO_N_SAMPLES, 1), dtype=tf.float32),
    tf.TensorSpec(shape=(None, N_FRAMES, 88),  dtype=tf.float32),
    tf.TensorSpec(shape=(None, N_FRAMES, 88),  dtype=tf.float32),
    tf.TensorSpec(shape=(None, N_FRAMES, 264), dtype=tf.float32),
])
def train_step(audio, t_onset, t_note, t_contour):
    with tf.GradientTape() as tape:
        out = saved_model(audio, training=True)
        loss = tf.reduce_mean(
            transcription_loss(t_onset,   out['onset'],   DEFAULT_LABEL_SMOOTHING) +
            transcription_loss(t_note,    out['note'],    DEFAULT_LABEL_SMOOTHING) +
            transcription_loss(t_contour, out['contour'], DEFAULT_LABEL_SMOOTHING))
    grads = tape.gradient(loss, saved_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, saved_model.trainable_variables))
    return loss

def run_training(train_records, val_records, cfg):
    os.makedirs(cfg.out_dir, exist_ok=True)
    train_ds = build_tf_dataset(train_records, cfg, augment=True,  shuffle=True)
    val_ds   = build_tf_dataset(val_records,   cfg, augment=False, shuffle=False)

    step = 0; best_val = float('inf')
    train_iter = iter(train_ds)

    while step < cfg.total_steps:
        try: audio, t_on, t_note, t_cont = next(train_iter)
        except StopIteration:
            train_iter = iter(train_ds)
            audio, t_on, t_note, t_cont = next(train_iter)

        loss = train_step(audio, t_on, t_note, t_cont)
        step += 1

        if step % cfg.val_every == 0:
            val_losses = []
            for a2, t_on2, t_n2, t_c2 in val_ds.take(20):
                out2 = saved_model(a2, training=False)
                val_losses.append(float(tf.reduce_mean(
                  transcription_loss(t_on2, out2['onset'],   DEFAULT_LABEL_SMOOTHING) +
                  transcription_loss(t_n2,  out2['note'],    DEFAULT_LABEL_SMOOTHING) +
                  transcription_loss(t_c2,  out2['contour'], DEFAULT_LABEL_SMOOTHING)
              )))
            val_loss = np.mean(val_losses)
            print(f"step {step:>6} | train loss {float(loss):.4f} | val loss {float(val_loss):.4f}")
            if val_loss < best_val:
                best_val = val_loss
                tf.saved_model.save(saved_model, os.path.join(cfg.out_dir, 'best_model'))
                print(f"  -> saved (best val loss {best_val:.4f})")

    print(f"Training done. Best val loss: {best_val:.4f}")
    return saved_model

print("Training cells ready.")

DEFAULT_LABEL_SMOOTHING: 0.2
Optimizer ready | initial lr: 1e-05 | decay: 0.9 every 10000 steps
Training cells ready.


In [64]:
records = build_index(cfg)
print(f"GuitarSet: {sum(1 for r in records if r['source']=='guitarset')} | EGDB: {sum(1 for r in records if r['source']=='egdb')}")

splits = make_splits(records, held_out_players=("05",))
print(f"Train GS: {len(splits['guitarset_train'])} | Val GS: {len(splits['guitarset_val'])} | Test: {len(splits['test_guitarset'])}")

# Start with acoustic-only first (faster to confirm the loop works end-to-end)
train_recs, val_recs = variant_records(splits, "acoustic")
print(f"Training on {len(train_recs)} recordings, validating on {len(val_recs)}")

GuitarSet: 360 | EGDB: 0
Train GS: 240 | Val GS: 60 | Test: 60
Training on 240 recordings, validating on 60


In [46]:
# Smoke test: 100 steps only
cfg.total_steps = 100
cfg.val_every = 50
finetuned = run_training(train_recs, val_recs, cfg)

step     50 | train loss 0.9909 | val loss 0.9966
  -> saved (best val loss 0.9966)
step    100 | train loss 1.0108 | val loss 0.9962
  -> saved (best val loss 0.9962)
Training done. Best val loss: 0.9962


In [60]:
# Peek at one batch
train_ds = build_tf_dataset(train_recs, cfg, augment=False, shuffle=False)
audio_batch, t_on, t_note, t_cont = next(iter(train_ds))
print("audio shape:", audio_batch.shape)       # should be (4, 43844, 1)
print("onset shape:", t_on.shape)              # should be (4, 172, 88)
print("non-zero onset frames:", tf.reduce_sum(tf.cast(t_on > 0, tf.float32)).numpy())

audio shape: (4, 43844, 1)
onset shape: (4, 172, 88)
non-zero onset frames: 31.0


In [65]:
import time
# First call will still trace
loss = train_step(audio, t_on, t_note, t_cont)
# Second call should be fast
t0 = time.time()
for _ in range(5):
    loss = train_step(audio, t_on, t_note, t_cont)
print(f"Avg step time: {(time.time()-t0)/5:.2f}s | loss: {float(loss):.4f}")

Avg step time: 2.16s | loss: 0.9859


In [66]:
cfg.total_steps = 5_000
cfg.val_every   = 500
finetuned = run_training(train_recs, val_recs, cfg)

step    500 | train loss 0.9956 | val loss 0.9925
  -> saved (best val loss 0.9925)


KeyboardInterrupt: 

In [1]:
!pip -q install piano_transcription_inference
!pip -q install torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.4 MB/s eta 0:00:00


In [2]:
import torch
from piano_transcription_inference import PianoTranscription
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

torch: 2.11.0+cu128
CUDA: True


## 10. Evaluation — decode notes + P50 / R50 / F50  ✅ *unit-tested metric*

Note-level multi-pitch F at ±50 ms, identical in spirit to your current 0.776 and TART's 0.838, so the comparison is apples-to-apples.

> `# >>> IMPL-SPECIFIC` `decode_notes` should call the port's posteriorgram->note decoder (thresholds from your tuned 0.40 amplitude / onset settings). If `mir_eval` is available, prefer `mir_eval.transcription.precision_recall_f1_overlap` with `offset_ratio=None`.

In [ ]:
def match_notes(gt, pred, onset_tol=0.05):
    """Greedy pitch+onset matching -> (P,R,F). Same matcher as the detection diagnostic."""
    cands=[]
    for pi,p in enumerate(pred):
        for gi,g in enumerate(gt):
            if int(p["midi"])!=int(g["midi"]): continue
            dt=abs(p["onset"]-g["onset"])
            if dt<=onset_tol: cands.append((dt,pi,gi))
    cands.sort(key=lambda x:x[0])
    up,ug=set(),set()
    for dt,pi,gi in cands:
        if pi in up or gi in ug: continue
        up.add(pi); ug.add(gi)
    tp=len(up); fp=len(pred)-tp; fn=len(gt)-tp
    P=tp/(tp+fp) if tp+fp else 0.0; R=tp/(tp+fn) if tp+fn else 0.0
    F=2*P*R/(P+R) if P+R else 0.0
    return P,R,F

# ---- metric self-test ----
_gt=[{"onset":0.0,"midi":60},{"onset":0.5,"midi":64},{"onset":1.0,"midi":67}]
_pr=[{"onset":0.02,"midi":60},{"onset":0.54,"midi":64},{"onset":1.0,"midi":69},{"onset":2.0,"midi":50}]
_P,_R,_F = match_notes(_gt,_pr)
assert (round(_P,3),round(_R,3))==(0.5,0.667); print("P50/R50/F50 metric OK")

def decode_notes(model, forward_heads, audio_path, cfg, device="cuda"):
    raise NotImplementedError("Wire to the port's posteriorgram->note decoder (use your tuned thresholds).")  # >>> IMPL-SPECIFIC

def evaluate(model, forward_heads, test_records, cfg, device="cuda"):
    rows=[]
    for r in test_records:
        gt = notes_for(r)
        pred = decode_notes(model, forward_heads, r["audio"], cfg, device)
        P,R,F = match_notes(gt, pred, cfg.onset_tol_sec)
        rows.append({"id":r["id"],"source":r["source"],"P50":P,"R50":R,"F50":F,"n_gt":len(gt)})
    df=pd.DataFrame(rows)
    # note-weighted aggregate (comparable to your summary table)
    agg = {"P50":np.average(df.P50,weights=df.n_gt),
           "R50":np.average(df.R50,weights=df.n_gt),
           "F50":np.average(df.F50,weights=df.n_gt)}
    return df, agg

def quick_val_f50(model, forward_heads, val_loader, cfg, device):
    return 0.0  # >>> IMPL-SPECIFIC: decode a val subset and return mean F50 (used for checkpointing)

## 11. Experiment plan — reproduce TART's comparison

Run the same three variants and report each on **both** held-out GuitarSet and an OOD set, so you can see the robustness story, not just one number.

| Variant | Fine-tuned on | TART GuitarSet F50 | TART EGDB F50 |
|---|---|---|---|
| Base (no FT) | — | 0.704 | 0.596 |
| Acoustic | GuitarSet | 0.838 | (drops off-domain) |
| Electric | EGDB | — | 0.752 |
| **Acoustic-Electric** | **GuitarSet+EGDB** | **0.838** | **0.779** |

**Steps:**
1. Establish the **Base** number first: run `evaluate(...)` on your held-out test with the *un-fine-tuned* Basic Pitch. That's your real starting line (should land near your current 0.776 on GuitarSet).
2. Fine-tune **acoustic** and **acoustic_electric** variants; evaluate each on held-out GuitarSet **and** an OOD set (EGDB val, or the GM-style record-to-GuitarPro set).
3. Pick the winner on **validation**; report the held-out test number **once**.
4. Re-run the detection diagnostic on the fine-tuned model to confirm where it improved (polyphony recall? onset fragmentation?).

In [ ]:
def run_variant(variant, splits, cfg, device="cuda"):
    train_recs, val_recs = variant_records(splits, variant)
    tr = DataLoader(GuitarFinetuneDataset(train_recs, cfg, augment=True),
                    batch_size=cfg.batch_size, shuffle=True, num_workers=2, drop_last=True)
    va = DataLoader(GuitarFinetuneDataset(val_recs, cfg, augment=False),
                    batch_size=cfg.batch_size, shuffle=False, num_workers=2)
    # model, forward_heads = load_basic_pitch_trainable(cfg)   # >>> IMPL-SPECIFIC (cell 8)
    # model = train(model, forward_heads, tr, va, cfg, device)
    # df, agg = evaluate(model, forward_heads, splits["test_guitarset"], cfg, device)
    # return agg
    raise NotImplementedError("Fill in cells 7-10 IMPL-SPECIFIC hooks, then run.")

# Typical flow:
# records = build_index(cfg)
# splits  = make_splits(records, held_out_players=("05",))
# base_df, base_agg = evaluate(base_model, base_forward, splits["test_guitarset"], cfg)  # starting line
# ae_agg = run_variant("acoustic_electric", splits, cfg)
print("Plan ready. Fill the IMPL-SPECIFIC hooks (cells 7, 8, 10), then run_variant('acoustic_electric', ...).")

---
### Summary of what's solid vs. what needs your environment
- **Solid & tested:** segmentation, pitch-shift label math, target construction shape, player-disjoint splits, the P50/R50/F50 matcher, the experiment harness.
- **You wire in (`# >>> IMPL-SPECIFIC`):** the trainable Basic Pitch model + its input featurizer (cells 7–8), and the posteriorgram→note decoder (cell 10). These are the 3 hooks that depend on which Basic Pitch implementation you load.
- **Config is locked to TART's proven recipe** — 10 s / 1 s hop, ±2 semitone, batch 4, lr 1e-5 ×0.9/10k, 100k steps, **no onset jitter**, **GuitarSet+EGDB combined**.
